In [ ]:
// Urban Heat Islands Calculate Using Landsat 8 Satellite image
// Import Country boundary
var countries = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level2")
Map.addLayer(countries, {}, 'Global Country Boundary', false)

var point = [89.00146141096812, 23.913972653154357]
var centerCity = ee.Geometry.Point(point)
var kushtia = countries.filterBounds(centerCity)
             .map(function (feature){
               return feature.simplify(1000) // Using .simplify for skipping complexity
             })

Map.centerObject(kushtia,8)
Map.addLayer(kushtia, {}, 'Kushtia', false)

// Import Landsat-8 Satellite Data
var L8 = ee.ImageCollection("LANDSAT/LC08/C02/T1_TOA")
        .filterBounds(kushtia)
        .filter(ee.Filter.lt('CLOUD_COVER', 10))
        .filterDate('2024-03-01','2024-04-30')
        .median()
        .clip(kushtia)
        
print(L8) 
Map.addLayer(L8, imageVisParam,'False Color Composite')// Using false color composite Band 5,4,3

//Compute NDVI
var NDVI = L8.normalizedDifference(['B5','B4']).rename('NDVI');

Map.addLayer(NDVI, {
  min: -0.05527440044071946,
  max: 0.6375621943052768,
  palette: ['blue','white','green']
}, 'NDVI');

//Calculate Maximum value
// Calculate Maximum NDVI value
var max = NDVI.reduceRegion({
  reducer: ee.Reducer.max(),
  geometry: kushtia,
  scale: 30,
  maxPixels: 1e9
}).get('NDVI');

print('Max NDVI:', max);
  
print(max)

// Calculate Minimum NDVI value
var min = NDVI.reduceRegion({
  reducer: ee.Reducer.min(),
  geometry: kushtia,
  scale: 30,
  maxPixels: 1e9
}).get('NDVI');

print('Min NDVI:', min);
  
print(min)
//Surface emissivity formula for LST pv= vegetation proportion
// Convert min and max to ee.Number
var minNDVI = ee.Number(min);
var maxNDVI = ee.Number(max);

// Proportion of vegetation (Pv)
var pv = NDVI.subtract(minNDVI)
             .divide(maxNDVI.subtract(minNDVI))
             .pow(2)
             .rename('Pv');

Map.addLayer(pv, {
  min: 0,
  max: 1,
  palette: ['white', 'green']
}, 'Proportion Vegetation');

//Calculate emissivity
var emis = pv.multiply(ee.Number(0.004)).add(ee.Number(0.986)).rename('Emissivity')
Map.addLayer(emis,  {
  min: 0.9863298962569538,
  max: 0.9889421355831213,
  palette:['blue','green','yellow','red']},'Emissivisty');
  
//Calculate LST = BT / (1 + (λ * BT / c2) * ln(ε))
// Brightness Temperature BT
var BT =L8.select('B10').rename('BT')

Map.addLayer(BT, {
  min: 290,
  max: 320,
  palette: ['blue', 'green', 'yellow', 'red']
}, 'Brightness Temperature');

var LST = BT.expression('(BT /(1+(0.00115*(BT/1.438))*log(emis)))-273',{
  'BT': BT,
  'emis': emis,
}).rename('LST')

Map.addLayer(LST,{
  min:27.518004098552463,
  max:40.314991857537116,
  palette: ['blue', 'cyan', 'green', 'yellow', 'orange', 'red']
}, 'LST');

//Urban Heat Island 
// Equation = Ts - Tm / STD   where, Ts = LST, Tm = Mean of LST and STD = Standard deviation

     //Calculate Mean and STD of LST
var mean = LST.reduceRegion({
  reducer: ee.Reducer.mean(),
  geometry: kushtia,
  scale: 30,
  maxPixels: 1e9
}).get('LST');

print('Mean LST:', mean);

var std = LST.reduceRegion({
  reducer: ee.Reducer.stdDev(),
  geometry: kushtia,
  scale: 30,
  maxPixels: 1e9
}).get('LST');

print('STD LST:', std);

// Convert to ee.Number
var meanLST = ee.Number(mean);
var stdLST = ee.Number(std);

var UHI = LST.subtract(meanLST).divide(stdLST)
Map.addLayer(UHI, {
  min:-2.071506254249786,
  max: 2.9246336516700415,
  palette: ['blue', 'cyan', 'green', 'yellow', 'orange', 'red']
}, 'Urban Heat Island');

// Export to Drive 
Export.image.toDrive({
  image: UHI,
  description: 'Urban Heat Island',
  folder: 'GEE_Exports',
  fileNamePrefix: 'UHI_Kushtia_2024',
  region: kushtia.geometry(),
  scale: 30,
  crs: 'EPSG:4326',
  maxPixels: 1e13,
  fileFormat: 'GeoTIFF'
});

